In [1]:
import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [3]:
folders = [
    "models",
    "outputs",
    "outputs/tables"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [5]:
df = pd.read_csv("data/customer_queries.csv")

df.head()

,query,intent,clean_query
0,I submitted my claim last week but I have not ...,claim_enquiry,i submitted my claim last week but i have not ...
1,"Good day, I submitted my claim last week but I...",claim_enquiry,good day i submitted my claim last week but i ...
2,"Hi, I submitted my claim last week but I have ...",claim_enquiry,hi i submitted my claim last week but i have n...
3,"Hello, I submitted my claim last week but I ha...",claim_enquiry,hello i submitted my claim last week but i hav...
4,"Please assist, I submitted my claim last week ...",claim_enquiry,please assist i submitted my claim last week b...


In [7]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns)
print("\nIntent distribution:")
print(df["intent"].value_counts())

Dataset shape: (360, 3)

Columns:
Index(['query', 'intent', 'clean_query'], dtype='object')

Intent distribution:
intent
claim_enquiry          60
policy_update          60
benefit_question       60
complaint              60
payment_issue          60
general_information    60
Name: count, dtype: int64


In [9]:
X = df["clean_query"]
y = df["intent"]

print("Number of text samples:", len(X))
print("Number of intent categories:", y.nunique())

Number of text samples: 360
Number of intent categories: 6


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 288
Testing samples: 72


In [13]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", MultinomialNB())
    ]),

    "Support Vector Machine": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", LinearSVC(
            random_state=42
        ))
    ])
}

In [15]:
results = []
trained_models = {}

for model_name, model_pipeline in models.items():
    print(f"\nTraining model: {model_name}")
    
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    cv_scores = cross_val_score(
        model_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    
    results.append({
        "model": model_name,
        "test_accuracy": accuracy,
        "mean_cv_accuracy": cv_scores.mean(),
        "std_cv_accuracy": cv_scores.std()
    })
    
    trained_models[model_name] = model_pipeline
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
    print(f"CV Std Dev: {cv_scores.std():.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


Training model: Logistic Regression
Test Accuracy: 1.0000
Mean CV Accuracy: 0.9930
CV Std Dev: 0.0140

Classification Report:
                     precision    recall  f1-score   support

   benefit_question       1.00      1.00      1.00        12
      claim_enquiry       1.00      1.00      1.00        12
          complaint       1.00      1.00      1.00        12
general_information       1.00      1.00      1.00        12
      payment_issue       1.00      1.00      1.00        12
      policy_update       1.00      1.00      1.00        12

           accuracy                           1.00        72
          macro avg       1.00      1.00      1.00        72
       weighted avg       1.00      1.00      1.00        72


Training model: Naive Bayes
Test Accuracy: 1.0000
Mean CV Accuracy: 0.9930
CV Std Dev: 0.0140

Classification Report:
                     precision    recall  f1-score   support

   benefit_question       1.00      1.00      1.00        12
      claim_enquir

In [17]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="test_accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,model,test_accuracy,mean_cv_accuracy,std_cv_accuracy
0,Logistic Regression,1.0,0.992982,0.014035
1,Naive Bayes,1.0,0.992982,0.014035
2,Support Vector Machine,1.0,0.996491,0.007018


In [19]:
results_df.to_csv("outputs/tables/model_comparison_results.csv", index=False)

print("Model comparison table saved successfully.")

Model comparison table saved successfully.


In [21]:
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Logistic Regression


In [23]:
model_path = "models/best_intent_classifier.pkl"

joblib.dump(best_model, model_path)

print(f"Best model saved to: {model_path}")

Best model saved to: models/best_intent_classifier.pkl


In [25]:
new_queries = [
    "I want to know if my claim has been approved.",
    "Please update my banking details.",
    "What dental benefits are available on my plan?",
    "I am unhappy with the service I received.",
    "My debit order failed this month.",
    "How do I register on the mobile app?"
]

predictions = best_model.predict(new_queries)

prediction_df = pd.DataFrame({
    "customer_query": new_queries,
    "predicted_intent": predictions
})

prediction_df

,customer_query,predicted_intent
0,I want to know if my claim has been approved.,claim_enquiry
1,Please update my banking details.,policy_update
2,What dental benefits are available on my plan?,benefit_question
3,I am unhappy with the service I received.,complaint
4,My debit order failed this month.,payment_issue
5,How do I register on the mobile app?,general_information


In [27]:
prediction_df.to_csv("outputs/tables/example_predictions.csv", index=False)

print("Example predictions saved successfully.")

Example predictions saved successfully.
